# 1D CNN for Sequence Labeling
## 6-Class Word Labeling with PyTorch

## Cell 1: Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device detection: Apple Silicon (MPS) > CUDA (NVIDIA) > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f"Using device: {device} (Apple Silicon GPU)")
    print(f"MPS built: {torch.backends.mps.is_built()}")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using device: {device} (NVIDIA GPU)")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print(f"Using device: {device}")
    print("Note: GPU acceleration not available. Training will be slower.")

print(f"PyTorch version: {torch.__version__}")

## Cell 2: Data Segmentation
### Prepare and segment your sequence data

In [ ]:
# Example: Generate synthetic data for demonstration
# Replace this with your actual data loading code

# Parameters
num_samples = 1000
sequence_length = 100
num_features = 1  # For 1D sequences

# Generate synthetic sequences (replace with your actual data)
# Each sequence is a 1D array of length 100
X = np.random.randn(num_samples, sequence_length, num_features).astype(np.float32)

# For demonstration: create pattern-based synthetic data
# You would load your actual segmented data here
for i in range(num_samples):
    # Add some patterns to make classification easier
    pattern_type = i % 6
    if pattern_type == 0:
        X[i, :20] += 2.0  # Pattern for class 0
    elif pattern_type == 1:
        X[i, 20:40] += 2.0  # Pattern for class 1
    elif pattern_type == 2:
        X[i, 40:60] += 2.0  # Pattern for class 2
    elif pattern_type == 3:
        X[i, 60:80] += 2.0  # Pattern for class 3
    elif pattern_type == 4:
        X[i, 80:] += 2.0  # Pattern for class 4
    else:
        X[i, ::10] += 2.0  # Pattern for class 5

print(f"Data shape: {X.shape}")
print(f"Sequence length: {sequence_length}")
print(f"Number of features: {num_features}")

# Visualization of sample sequences
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    ax.plot(X[i].flatten())
    ax.set_title(f'Sample {i} (Class {i % 6})')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')
plt.tight_layout()
plt.show()

## Cell 3: Label Assignment & Train/Test Split
### Assign labels to 6 word categories and split data 80/20

In [ ]:
# Define the 6 word labels
word_labels = ['yes', 'no', 'wait', 'bye', 'help', 'pain']
num_classes = len(word_labels)

# Generate labels (replace with your actual labels)
y = np.array([i % num_classes for i in range(num_samples)])

print(f"Number of classes: {num_classes}")
print(f"Class labels: {word_labels}")
print(f"Label distribution:")
for i, label in enumerate(word_labels):
    count = np.sum(y == i)
    print(f"  {label}: {count} samples ({count/len(y)*100:.1f}%)")

# 80/20 Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")
print(f"Train/Test ratio: {len(X_train)/len(X_test):.1f}")

# Create PyTorch Dataset
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = SequenceDataset(X_train, y_train)
test_dataset = SequenceDataset(X_test, y_test)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nBatch size: {batch_size}")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

## Cell 4: 1D CNN Model Definition and Training

In [ ]:
# Define 1D CNN Model
class CNN1D(nn.Module):
    def __init__(self, input_channels=1, num_classes=6, sequence_length=100):
        super(CNN1D, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        
        # Calculate the size after convolutions and pooling
        # sequence_length -> /2 -> /2 -> /2 = sequence_length / 8
        flattened_size = 128 * (sequence_length // 8)
        
        # Fully connected layers
        self.fc1 = nn.Linear(flattened_size, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # Input shape: (batch_size, sequence_length, input_channels)
        # Conv1d expects: (batch_size, input_channels, sequence_length)
        x = x.permute(0, 2, 1)
        
        # Convolutional blocks
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Initialize model
model = CNN1D(input_channels=num_features, num_classes=num_classes, sequence_length=sequence_length)
model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Print model summary
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

## Cell 5: Training Loop

In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# Validation function
def validate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(test_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# Training loop
num_epochs = 50
train_losses = []
train_accs = []
test_losses = []
test_accs = []

print("Starting training...\n")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = validate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
        print()

print("Training completed!")
print(f"Final Test Accuracy: {test_accs[-1]:.2f}%")

## Cell 6: Results Visualization and Evaluation

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(train_accs, label='Train Accuracy')
ax2.plot(test_accs, label='Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Get predictions for test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=word_labels))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=word_labels, yticklabels=word_labels)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Cell 7: Save Model (Optional)

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'test_losses': test_losses,
    'train_accs': train_accs,
    'test_accs': test_accs,
    'word_labels': word_labels
}, 'cnn1d_model.pth')

print("Model saved as 'cnn1d_model.pth'")

# To load the model later:
# checkpoint = torch.load('cnn1d_model.pth')
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])